In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
import re
import string
from collections import Counter
import gc
import numpy as np
from opacus import PrivacyEngine
from opacus.layers import DPLSTM
from privacy_analysis.compute_privacy_sgm import compute_dp_sgd_privacy

/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
imdb = load_dataset("imdb")
s140 = load_dataset("adilbekovich/Sentiment140Twitter")

max_imdb_contribution = len(imdb["train"])
max_s140_contribution = len(s140["train"])

total_if_imdb_maxed = int(len(imdb["train"]) / 0.96)
total_if_s140_maxed = int(len(s140["train"]) / 0.04)
total_desired_size = min(total_if_imdb_maxed, total_if_s140_maxed)
imdb_size = int(0.96 * total_desired_size)
s140_size = int(0.04 * total_desired_size)

np.random.seed(42)
imdb_indices = np.random.choice(len(imdb["train"]), size=len(imdb), replace=False)
s140_indices = np.random.choice(len(s140["train"]), size=6250, replace=False)

imdb_subset = imdb["train"]
s140_subset = s140["train"].select(s140_indices)

In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

counter = Counter()
for example in imdb["train"]:
    counter.update(tokenize(example["text"]))

min_freq = 3
vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in counter.items():
    if freq >= min_freq:
        vocab[word] = len(vocab)
inv_vocab = {v: k for k, v in vocab.items()}
PAD_IDX = vocab["<pad>"]
UNK_IDX = vocab["<unk>"]

def text_pipeline(text, max_len=256):
    tokens = tokenize(text)[:max_len]
    bow_vector = torch.zeros(len(vocab), dtype=torch.float)
    for token in tokens:
        idx = vocab.get(token, UNK_IDX)
        bow_vector[idx] += 1
    return bow_vector

In [4]:
class TextDataset(Dataset):
    def __init__(self, hf_dataset, label_transform, text_field, max_len=256):
        self.data = hf_dataset
        self.label_transform = label_transform
        self.max_len = max_len
        self.text_field = text_field

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = text_pipeline(item[self.text_field], max_len=self.max_len)
        label = torch.tensor(self.label_transform(item["label"]), dtype=torch.long)
        return tokens, label

def collate_batch(batch):
    texts, labels = zip(*batch)
    padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(labels)

In [5]:
train_loader_public = DataLoader(
    TextDataset(s140_subset, label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=True, collate_fn=collate_batch
)

train_loader_private = DataLoader(
    TextDataset(imdb_subset, label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=True, collate_fn=collate_batch
)

test_loader = DataLoader(
    TextDataset(imdb["test"], label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=False, collate_fn=collate_batch
)

In [6]:
class Model(nn.Module):
    def __init__(self, vocab_size, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(vocab_size, num_classes)

    def forward(self, x):
        return self.fc(x)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(len(vocab), num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
privacy_engine = PrivacyEngine()
gc.collect()

/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


50

In [8]:
def DOPESGD(model, D_loader, D_s_loader, test_loader, lr, sigma, C, l, T, delta):
    """
    Parameters:
        D_loader: private training data loader
        n: private batch size
        D_s_loader: public training data loader
        n_s: public batch size
        lr: learning rate
        sigma: noise scale
        C: gradient norm clip
        l: loss function
        T: number of training iterations
    """

    train_losses = []
    train_accuracies = []
    test_losses = []
    test_accuracies = []

    n_private = len(D_loader.dataset)
    batch_size_private = D_loader.batch_size
    
    optim = torch.optim.SGD(model.parameters(), lr)
    pub_iter = iter(D_s_loader)

    for epoch in range(T):
        
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for priv_inputs, priv_labels in D_loader:
            
            try:
                pub_inputs, pub_labels = next(pub_iter)
            except StopIteration:
                pub_iter = iter(D_s_loader)
                pub_inputs, pub_labels = next(pub_iter)
            
            optim.zero_grad()
            pub_out = model(pub_inputs)
            pub_loss = l(pub_out, pub_labels)
            pub_loss.backward()
            pub_grad = []
            for p in model.parameters():
                pub_grad.append(p.grad.detach().clone())
            pub_grad = [g.clone() for g in pub_grad]

            grads = [torch.zeros_like(p) for p in model.parameters()]

            for x, y in zip(priv_inputs, priv_labels):
                optim.zero_grad()
                output = model(x.unsqueeze(0))
                loss = l(output, y.unsqueeze(0))
                loss.backward()

                for i, p in enumerate(model.parameters()):
                    if p.requires_grad and p.grad is not None:
                        grads[i] += pub_grad[i] + ((p.grad - pub_grad[i]) * C) / (torch.max(torch.tensor(C), torch.norm(p.grad - pub_grad[i])))

                for i in range(len(grads)):    
                    grads[i] += torch.normal(0, sigma * C, grads[i].shape)

            for p, g in zip(model.parameters(), grads):
                if p.requires_grad:
                    p.grad = g / len(priv_inputs)
            optim.step()

            total_loss += pub_loss.item() * priv_inputs.size(0)
            with torch.no_grad():
                priv_out = model(priv_inputs)
                _, predicted = torch.max(priv_out.data, 1)
                total += priv_labels.size(0)
                correct += (predicted == priv_labels).sum().item()
        
        epoch_loss = total_loss / total
        epoch_accuracy = 100 * correct / total

        model.eval()
        correct = 0
        total = 0
        test_loss = 0.0

        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = model(inputs)
                loss = l(outputs, labels)
                test_loss += loss.item() * labels.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        test_accuracy = 100 * correct / total
        test_loss = test_loss / total

        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_accuracy)
        test_losses.append(test_loss)
        test_accuracies.append(test_accuracy)

        eps, _ = compute_dp_sgd_privacy(n_private, batch_size_private, sigma, epoch + 1, delta)

        print(f"Epoch [{epoch+1}/{T}] ... Training Loss: {epoch_loss:.4f} ... Training Accuracy: {epoch_accuracy:.2f}% ... Test Loss: {test_loss:.4f} ... Test Accuracy: {test_accuracy:.2f}% ... Epsilon: {eps:.4f} ... Delta: {delta}")

    return train_losses, train_accuracies, test_losses, test_accuracies

In [9]:
train_losses, train_accuracies, test_losses, test_accuracies = DOPESGD(
                                                                        model=model,
                                                                        D_loader=train_loader_private,
                                                                        D_s_loader=train_loader_public,
                                                                        test_loader=test_loader,
                                                                        lr=0.01, 
                                                                        sigma=0.69, 
                                                                        C=1.0, 
                                                                        l=criterion,
                                                                        T=10,
                                                                        delta=1e-5
                                                                      )

DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.69 iterated over 391 steps satisfies differential privacy with eps = 2.15 and delta = 1e-05.
The optimal RDP order is 5.5.
Epoch [1/10] ... Training Loss: 0.6696 ... Training Accuracy: 58.44% ... Test Loss: 0.8037 ... Test Accuracy: 59.44% ... Epsilon: 2.1542 ... Delta: 1e-05
DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.69 iterated over 782 steps satisfies differential privacy with eps = 2.33 and delta = 1e-05.
The optimal RDP order is 5.5.
Epoch [2/10] ... Training Loss: 0.6665 ... Training Accuracy: 59.32% ... Test Loss: 1.1267 ... Test Accuracy: 57.39% ... Epsilon: 2.3295 ... Delta: 1e-05
DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.69 iterated over 1172 steps satisfies differential privacy with eps = 2.48 and delta = 1e-05.
The optimal RDP order is 5.0.
Epoch [3/10] ... Training Loss: 0.7439 ... Training Accuracy: 58.84% ... Test Loss: 1.6946 ... Test Accuracy: 55.69% ... Epsilon: 2.4772 ... D